In [ ]:
#!pip install fitter openpyxl
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
import datetime as dt

In [ ]:
df = pd.read_excel('concatplusplus.xlsx')
df.head()

,Station,Bus arrival time,Opening the doors,Closing the doors,Number of entries,Number of exits,Bus movement,position,Date,Time between Station,Seconds between
0,ولیعصر,06:49:58,06:50:07,06:50:17,2,7,06:50:20,شمال به جنوب,شنبه,0.001863,161
1,ولیعصر,06:51:40,06:51:42,06:51:48,0,0,06:51:51,شمال به جنوب,شنبه,0.001181,102
2,ولیعصر,07:00:12,07:00:25,07:00:37,15,4,07:00:40,شمال به جنوب,شنبه,0.005926,512
3,ولیعصر,07:01:16,07:01:26,07:01:35,3,0,07:01:39,شمال به جنوب,شنبه,0.000741,64
4,خاوران,06:48:14,06:48:23,06:50:59,26,0,06:51:02,جنوب به شمال,شنبه,0.000718,62


In [ ]:
position = ['جنوب به شمال','شمال به جنوب']
Time = ['صبح','ظهر','شب']
Station = ['خاوران','بسیج','رحمانی','ولیعصر']
results = []
dict_df = {}

In [ ]:
def get_position_code(position_name):
    if position_name == 'جنوب به شمال':
        return 'S'
    elif position_name == 'شمال به جنوب':
        return 'N'


In [ ]:
def get_Time_code(Time_name):
    if Time_name == 'صبح':
        return 'M'
    elif Time_name == 'ظهر':
        return 'A'
    elif Time_name == 'شب':
        return 'N'


In [ ]:
def get_Station_code(Station_name):
    if Station_name == 'خاوران':
        return 'K'
    elif Station_name == 'بسیج':
        return 'B'
    elif Station_name == 'رحمانی':
        return 'R'
    elif Station_name == 'ولیعصر':
        return 'V'

In [ ]:
KSM = df[(df['Station'] == 'خاوران') & (df['position'] == 'جنوب به شمال')]
KSM.head()

,Station,Bus arrival time,Opening the doors,Closing the doors,Number of entries,Number of exits,Bus movement,position,Date,Time between Station,Seconds between
4,خاوران,06:48:14,06:48:23,06:50:59,26,0,06:51:02,جنوب به شمال,شنبه,0.000718,62
5,خاوران,06:51:09,06:51:32,06:54:06,27,0,06:54:10,جنوب به شمال,شنبه,0.002025,175
6,خاوران,06:54:12,06:54:22,06:59:36,51,0,06:59:38,جنوب به شمال,شنبه,0.002118,183
7,خاوران,06:59:47,06:59:54,07:03:00,51,0,07:03:04,جنوب به شمال,شنبه,0.003877,335
34,خاوران,12:20:42,12:20:50,12:31:00,54,0,12:31:14,جنوب به شمال,شنبه,0.004190,362


In [ ]:
count = 0
for i in Station:
  for j in position :

      key = f"{i}_{j}"
      df_temp = df[(df['Station'] == i) & (df['position'] == j)]
      dict_df[key] = {'Name' : get_Station_code(i)+get_position_code(j),'DF':df_temp}
      count+=1

print(count)


8


In [ ]:
df.head()

,Station,Bus arrival time,Opening the doors,Closing the doors,Number of entries,Number of exits,Bus movement,position,Date,Time between Station,Seconds between
0,ولیعصر,06:49:58,06:50:07,06:50:17,2,7,06:50:20,شمال به جنوب,شنبه,0.001863,161
1,ولیعصر,06:51:40,06:51:42,06:51:48,0,0,06:51:51,شمال به جنوب,شنبه,0.001181,102
2,ولیعصر,07:00:12,07:00:25,07:00:37,15,4,07:00:40,شمال به جنوب,شنبه,0.005926,512
3,ولیعصر,07:01:16,07:01:26,07:01:35,3,0,07:01:39,شمال به جنوب,شنبه,0.000741,64
4,خاوران,06:48:14,06:48:23,06:50:59,26,0,06:51:02,جنوب به شمال,شنبه,0.000718,62


In [ ]:
distributions = ['norm', 'expon', 'lognorm', 'gamma', 'beta', 'uniform']

In [ ]:
dict = {
    'name':[],
    'dist_name':[],
    'd_statistic':[],
    'ks_p_value':[],
    'mean':[],
    'variance':[],
    'std':[]
}

In [ ]:
def get_input(Column_Name):
  for i,j in dict_df.items():
    pow = (j['DF'][Column_Name],j['Name'])
    for dist_name in distributions:
      try:
        dist = getattr(stats, dist_name)  # گرفتن شیء توزیع از scipy.stats
        params = dist.fit(pow[0])            # برازش پارامترها به داده‌ها
        D, p = stats.kstest(pow[0], dist_name, args=params)  # انجام آزمون K-S
        dict['name'].append(pow[1])
        dict['dist_name'].append(dist_name)
        dict['d_statistic'].append(D)
        dict['ks_p_value'].append(p)
        dict['mean'].append(pow[0].mean())
        dict['variance'].append(pow[0].var())
        dict['std'].append(pow[0].std())
      except Exception as e:
          continue
    lambda_poisson = np.mean(pow[0])
    data_sorted = np.sort(pow[0])
    ecdf = np.arange(1, len(data_sorted) + 1) / len(data_sorted)
    poisson_cdf = stats.poisson.cdf(data_sorted, lambda_poisson)
    D_poisson = np.max(np.abs(ecdf - poisson_cdf))
    dict['name'].append(pow[1])
    dict['dist_name'].append('poisson')
    dict['d_statistic'].append(D_poisson)
    dict['ks_p_value'].append('-')
    dict['mean'].append(pow[0].mean())
    dict['variance'].append(pow[0].var())
    dict['std'].append(pow[0].std())

 #   plt.hist(pow[0], bins=np.arange(pow[0].min(), pow[0].max()+2)-0.5, density=True, alpha=0.6, color='g', label=f'Data histogram of {pow[1]}')
 #   x = np.arange(pow[0].min(), pow[0].max()+1)
  #  plt.plot(x, stats.poisson.pmf(x, lambda_poisson), '-b', ms=8, label=f'Poisson of {pow[1]}')
  #  plt.xlabel('Value')
  #  plt.ylabel('Probability')
  #  plt.title(f'Data histogram and Poisson for {Column_Name} {pow[1]}')
  #  plt.savefig(f'{pow[1]}.png')
  #  plt.close()





In [ ]:
get_input('Number of entries')

/usr/local/lib/python3.11/dist-packages/scipy/stats/_distn_infrastructure.py:2129: RuntimeWarning: invalid value encountered in divide
  x = np.asarray((x - loc)/scale, dtype=dtyp)
/usr/local/lib/python3.11/dist-packages/scipy/stats/_continuous_distns.py:6921: RuntimeWarning: invalid value encountered in divide
  return np.sum((1 + np.log(shifted/scale)/shape**2)/shifted)
/usr/local/lib/python3.11/dist-packages/scipy/stats/_distn_infrastructure.py:409: RuntimeWarning: invalid value encountered in scalar divide
  return m3 / np.power(m2, 1.5)
/usr/local/lib/python3.11/dist-packages/scipy/stats/_distn_infrastructure.py:418: RuntimeWarning: invalid value encountered in scalar divide
  return m4 / m2**2 - 3
/usr/local/lib/python3.11/dist-packages/scipy/stats/_continuous_distns.py:800: RuntimeWarning: The iteration is not making good progress, as measured by the 
 improvement from the last ten iterations.
  a, b = optimize.fsolve(func, (1.0, 1.0))
/usr/local/lib/python3.11/dist-packages/sci

In [ ]:
get_input('Number of exits')

In [ ]:
get_input('Seconds between')

/usr/local/lib/python3.11/dist-packages/scipy/stats/_continuous_distns.py:795: RuntimeWarning: invalid value encountered in sqrt
  sk = 2*(b-a)*np.sqrt(a + b + 1) / (a + b + 2) / np.sqrt(a*b)


In [ ]:
dataframe1 = pd.DataFrame(dict)

In [ ]:
dataframe1.head()

,name,dist_name,d_statistic,ks_p_value,mean,variance,std
0,KS,norm,0.138425,0.789123,276.05,19029.628947,137.947921
1,KS,expon,0.260166,0.110627,276.05,19029.628947,137.947921
2,KS,lognorm,0.097338,0.981627,276.05,19029.628947,137.947921
3,KS,gamma,0.091023,0.990954,276.05,19029.628947,137.947921
4,KS,beta,0.091011,0.990968,276.05,19029.628947,137.947921


In [ ]:
dataframe1.to_excel('Info.xlsx',index=False)

In [ ]:
#f = Fitter(entries)
#f.fit()
#f.summary()